In [ ]:
%load_ext autoreload
%autoreload 2
import requests
import pandas as pd
import numpy as np
import holidays
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from pymongo import MongoClient
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import os
import sys
sys.path.append('..')
from model_utils.prepare_data import *
from model_utils.lstm_model import *


I0000 00:00:1784599779.464087  149020 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
%%time
# get the data we want to use
# df = fetch_traffic_data()
df = pd.read_csv('../data/crz_data_pull_20260720.csv',index_col=None)

CPU times: user 5.78 s, sys: 4.85 s, total: 10.6 s
Wall time: 10.6 s


In [1]:
def load_traffic_data_from_db_chunked(batch_size=10000):
    """
    Load data from MongoDB in chunks to avoid memory overload
    """
    print("Connecting to MongoDB...")
    client = MongoClient(MONGO_CONNECTION_STRING)
    db = client['CRZ']
    collection = db['traffic_data']
    
    total_count = collection.count_documents({})
    print(f"Total records in MongoDB: {total_count}")
    
    # Load in batches
    chunks = []
    for i in range(0, total_count, batch_size):
        print(f"Loading records {i} to {i + batch_size}...")
        data = list(collection.find({}, {'_id': 0}).skip(i).limit(batch_size))
        chunks.append(pd.DataFrame(data))
    
    # Combine all chunks
    df_all = pd.concat(chunks, ignore_index=True)
    client.close()
    
    print(f"✓ Loaded {len(df_all)} records total")
    return df_all

In [ ]:
from dotenv import load_dotenv
import os
from pymongo import MongoClient

load_dotenv()
MONGO_CONNECTION_STRING = os.getenv("LOCAL_MONGO_CONNECTION_STRING")

client = MongoClient(MONGO_CONNECTION_STRING)
db = client['CRZ']
collection = db['traffic_data']

# Now you can use it in Python
print(f"{collection.count_documents({}):,}")

2


In [ ]:
try:
    client = MongoClient(MONGO_CONNECTION_STRING)
    db = client['CRZ']
    collection = db['traffic_data']

    # clear existing data
    print("Clearing existing data from MongoDB...")
    collection.delete_many({})

    print(f"Inserting {len(df):,} rows into MongoDB...")
    collection.insert_many(df.to_dict('records'))

    collection.create_index("toll_10_minute_block")
    collection.create_index("detection_region")
    collection.create_index("detection_group")

    client.close()
    print("Data succesfully stored in MongoDB")
except Exception as e:
    print(f"Error connection to MongoDB: {e}")

Clearing existing data from MongoDB...
Inserting 5,733,504 rows into MongoDB...


In [8]:
load_dotenv()

True

In [3]:
conn_str = os.getenv("LOCAL_MONGO_CONNECTION_STRING")
print(f"Connection String: {conn_str}")

Connection String: mongodb://localhost:27017/


In [ ]:
def prepare_data(df, detection_region=None):
    """
    Prepare data for LSTM modeling
    Parameters:
    - df_all: full dataframe
    - detection_region: specific region or None for all regions
    Returns:
    - data: aggregated timeseries dataframe
    """
    # Convert date columns
    df['toll_date'] = pd.to_datetime(df['toll_date'])
    df['hour_of_day'] = pd.to_numeric(df['hour_of_day'])
    df['minute_of_hour'] = pd.to_numeric(df['minute_of_hour'])
    df['crz_entries'] = pd.to_numeric(df['crz_entries'])
    highest_day = df['toll_date'].max()
    removal_date = highest_day - timedelta(days=14)
    # Filter data
    data = df[df['toll_date'] <= removal_date].copy()
    if detection_region:
        data = data[data['detection_region'] == detection_region]
    # Select relevant columns and group
    data = data[['toll_date','hour_of_day', 'minute_of_hour', 'toll_10_minute_block', 'day_of_week', 'crz_entries']].copy()
    # Weekend bool col
    data['weekend_ind'] = data['day_of_week'].isin(['Saturday','Sunday']).astype(int)
    # Add holiday col
    data['holiday_ind'] = data['toll_date'].apply(lambda x: 1 if x in holidays.US() else 0)
    # Aggregate by time components
    data = data.groupby(['toll_date','hour_of_day', 'minute_of_hour', 'toll_10_minute_block', 'weekend_ind', 'holiday_ind']).agg(
            {'crz_entries':'sum'}
        ).reset_index()
    data = data.sort_values('toll_10_minute_block').reset_index(drop=True)
    data = data[['toll_date','hour_of_day','minute_of_hour','toll_10_minute_block','holiday_ind','crz_entries']]
    # Split data
    train_data = data[data['toll_date'] <= removal_date].copy()
    test_data = data[data['toll_date'] > removal_date].copy()